In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Configurar mitigação de erros

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



> **Note:** A versão beta de um novo modelo de execução já está disponível. O modelo de execução direcionada oferece mais flexibilidade para personalizar seu fluxo de trabalho de mitigação de erros. Consulte o guia [Modelo de execução direcionada](/guides/directed-execution-model) para mais informações.


<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
As técnicas de mitigação de erros permitem que os usuários mitiguem erros de circuito
modelando o ruído do dispositivo no momento da execução. Isso tipicamente
resulta em overhead de pré-processamento quântico relacionado ao treinamento do modelo e
overhead de pós-processamento clássico para mitigar erros nos resultados brutos
usando o modelo gerado.

O primitivo Estimator suporta várias técnicas de mitigação de erros, incluindo [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation), [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne), [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec) e [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options). Consulte [Técnicas de mitigação e supressão de erros](error-mitigation-and-suppression-techniques) para uma explicação de cada uma. Ao usar os primitivos, você pode ativar ou desativar métodos individualmente. Consulte a seção [Configurações personalizadas de erro](#advanced-error) para detalhes.

> **Note:** O Sampler não suporta mitigação de erros, mas você pode usar o pacote [mthree](https://qiskit.github.io/qiskit-addon-mthree/) (mitigação de medição livre de matrizes) para realizar a mitigação de erros localmente.

O Estimator também suporta `resilience_level`. O nível de resiliência especifica quanta resiliência deve ser construída contra
erros. Níveis mais altos geram resultados mais precisos, ao custo de
tempos de processamento mais longos. Os níveis de resiliência podem ser usados para configurar o
trade-off custo/precisão ao aplicar mitigação de erros à sua consulta de
primitivo. A mitigação de erros reduz erros (viés) nos resultados ao processar
as saídas de uma coleção, ou ensemble, de circuitos relacionados. O
grau de redução de erros depende do método aplicado. O nível de resiliência
abstrai a escolha detalhada do método de mitigação de erros para permitir
que os usuários raciocinem sobre o trade-off custo/precisão adequado à
sua aplicação.

Dado isso, cada nível corresponde a um método ou métodos com
nível crescente de overhead de amostragem quântica para permitir que você experimente
diferentes trade-offs entre tempo e precisão. A tabela a seguir mostra quais
níveis e métodos correspondentes estão disponíveis para cada um dos
primitivos.

> **Info:** A mitigação de erros é específica para cada tarefa, portanto as técnicas que você pode
> aplicar variam dependendo se você está amostrando uma distribuição ou gerando
> valores esperados.

<span id="resilience-table"></span>

O Estimator suporta os seguintes níveis de resiliência. O Sampler não suporta níveis de resiliência.

| Nível de Resiliência | Definição                                                                                                          | Técnica                                                                              |
|----------------------|--------------------------------------------------------------------------------------------------------------------|--------------------------------------------------------------------------------------|
| 0                    | Sem mitigação                                                                                                      | Nenhuma                                                                              |
| 1 [Padrão]           | Custos mínimos de mitigação: Mitiga erros associados a erros de leitura                                            | Twirled Readout Error eXtinction (TREX) com twirling de medição                     |
| 2                    | Custos médios de mitigação. Tipicamente reduz o viés nos estimadores, mas não é garantidamente livre de viés.      | Nível 1 + Extrapolação de Ruído Zero (ZNE) e twirling de portas

> **Info:** Os níveis de resiliência estão atualmente em beta, portanto o overhead de amostragem e
> a qualidade da solução variam de circuito para circuito. Novos recursos,
> opções avançadas e ferramentas de gerenciamento serão lançados continuamente.
> Não é garantido que métodos específicos de mitigação de erros sejam
> aplicados em cada nível de resiliência.

## Configurar o Estimator com níveis de resiliência
Você pode usar os níveis de resiliência para especificar técnicas de mitigação de erros, ou pode definir técnicas personalizadas individualmente conforme descrito em [Configurações personalizadas de erro.](#advanced-error)

<details>
<summary>Nível de Resiliência 0</summary>

Nenhuma mitigação de erros é aplicada ao programa do usuário.

</details>

<details>
<summary>Nível de Resiliência 1</summary>

O nível 1 aplica **mitigação de erros de leitura** e **twirling de medição** por meio de uma técnica livre de modelo conhecida
como Twirled Readout Error eXtinction (TREX). Ela reduz o erro de medição
diagonalizando o canal de ruído associado à medição, invertendo
aleatoriamente qubits por meio de portas X imediatamente antes da medição. Um
termo de reescalonamento do canal de ruído diagonal é aprendido pelo
benchmarking de circuitos aleatórios inicializados no estado zero. Isso permite
que o serviço remova o viés dos valores esperados resultante do
ruído de leitura. Esta abordagem é descrita com mais detalhes em [Model-free
readout-error mitigation for quantum expectation
values](https://arxiv.org/abs/2012.09738).

</details>

<details>
<summary>Nível de Resiliência 2</summary>

O nível 2 aplica as **técnicas de mitigação de erros incluídas no nível 1** e também aplica **twirling de portas** e usa o **método de Extrapolação de Ruído Zero (ZNE)**. O ZNE computa um
valor esperado do observável para diferentes fatores de ruído
(estágio de amplificação) e, em seguida, usa os valores esperados medidos para
inferir o valor esperado ideal no limite de ruído zero (estágio de
extrapolação). Esta abordagem tende a reduzir erros nos valores esperados, mas
não é garantida para produzir um resultado sem viés.

![Esta imagem mostra um gráfico. O eixo x é rotulado como Fator de amplificação de ruído. O eixo y é rotulado como Valor esperado. Uma linha com inclinação ascendente é rotulada como Valor mitigado. Pontos próximos à linha são valores com ruído amplificado. Há uma linha horizontal logo acima do eixo X rotulada como Valor exato.](../docs/images/guides/configure-error-mitigation/resilience-2.svg "Ilustração do método ZNE")

O overhead deste método escala com o número de fatores de ruído. As
configurações padrão amostram o valor esperado em três fatores de ruído,
levando a um overhead de aproximadamente 3x ao empregar este nível de resiliência.

No Nível 2, o método TREX inverte aleatoriamente qubits por meio de portas X imediatamente antes da medição,
e inverte o bit medido correspondente se uma porta X foi aplicada. Esta abordagem é descrita com mais detalhes em [Model-free
readout-error mitigation for quantum expectation
values](https://arxiv.org/abs/2012.09738).

</details>

### Exemplo
A interface `EstimatorV2` permite que os usuários trabalhem de forma integrada com a variedade de
métodos de mitigação de erros para reduzir erros nos valores esperados de
observáveis. O código a seguir usa Extrapolação de Ruído Zero e mitigação de erros de leitura simplesmente
definindo `resilience_level 2`.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## Configurações personalizadas de erro
Você pode ativar e desativar métodos individuais de mitigação e supressão de erros, incluindo desacoplamento dinâmico, twirling de portas e medições, mitigação de erros de medição, PEC e ZNE. Consulte [Técnicas de mitigação e supressão de erros](error-mitigation-and-suppression-techniques) para uma explicação de cada um.

> **Note:** - Nem todas as opções estão disponíveis para ambos os primitivos. Consulte a [tabela de opções disponíveis](runtime-options-overview#options-table) para a lista de opções disponíveis.
> - Nem todos os métodos funcionam em conjunto em todos os tipos de circuitos. Consulte a [tabela de compatibilidade de recursos](runtime-options-overview#options-compatibility-table) para detalhes.

<Tabs>
  <TabItem value="Estimator" label="Estimator">
    ```python
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}")
print(f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}")
```
  </TabItem>

  <TabItem value="Sampler" label="Sampler">